In [109]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

from affine import Affine
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling


BASE_SHIFT_DIR = Path(r"E:/Proj1_Pfynwald_Data/HyPlant/original")
COREG_DIR_BASE = Path(r"E:/Proj1_Pfynwald_Data/HyPlant/coregistration")

STUDY_AREA_PATH = Path(
    r"E:/Proj1_Pfynwald_Data/General/studyarea.gpkg"
)

In [110]:
# Manual RESIDUAL corrections in metres:
#
# +dx = east
# -dx = west
# +dy = north
# -dy = south

MANUAL_CORRECTIONS = [
    {
        "product": "SIF_SFM",
        "acquisition": "20240613-PHY-1149-1340-L2-W",
        "dx": -3.0,
        "dy": -4.0,
        "reason": "Residual offset after automatic coregistration",
    },
    {
        "product": "SIF_iFLD",
        "acquisition": "20230617-PHY-1124-1360-L2-W",
        "dx": +2.0,
        "dy": +28.0,
        "reason": "Residual offset after automatic coregistration",
    },
    {
        "product": "SIF_iFLD",
        "acquisition": "20240613-PHY-1149-1340-L2-W",
        "dx": +3.0,
        "dy": 0.0,
        "reason": "Residual offset after automatic coregistration",
    },
]

In [111]:
results = []

for correction in MANUAL_CORRECTIONS:

    product = correction["product"]
    acquisition = correction["acquisition"]

    dx_manual = float(correction["dx"])
    dy_manual = float(correction["dy"])
    reason = correction.get("reason", "")

    # --------------------------------------------------------
    # Load automatic solution
    # --------------------------------------------------------

    summary_path = (
        COREG_DIR_BASE
        / product
        / "qc"
        / "coreg_summary.csv"
    )

    summary = pd.read_csv(summary_path)

    rows = summary.loc[
        summary["acquisition"] == acquisition
    ]

    if len(rows) != 1:
        raise ValueError(
            f"Expected exactly one solution for "
            f"{product} / {acquisition}, found {len(rows)}."
        )

    row = rows.iloc[0]

    if row["status"] == "FAILED":
        raise ValueError(
            f"Automatic registration failed for {acquisition}."
        )

    # --------------------------------------------------------
    # Automatic + manual residual shift
    # --------------------------------------------------------

    dx_auto = float(row["x_shift_map"])
    dy_auto = float(row["y_shift_map"])

    dx_final = dx_auto + dx_manual
    dy_final = dy_auto + dy_manual

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    src_path = (
        BASE_SHIFT_DIR
        / product
        / row["image"]
    )

    auto_shared = Path(row["shared_output"])

    out_dir = (
        COREG_DIR_BASE
        / product
        / "shared_grid_manual"
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out_path = (
        out_dir
        / f"{src_path.stem}_coreg_shared_manual.tif"
    )

    # --------------------------------------------------------
    # Apply FINAL shift to ORIGINAL raster
    # and resample once to the existing shared grid
    # --------------------------------------------------------

    with rasterio.open(src_path) as src, \
         rasterio.open(auto_shared) as template:

        shifted_transform = (
            Affine.translation(
                dx_final,
                dy_final,
            )
            * src.transform
        )

        profile = template.profile.copy()
        nodata = template.nodata

        # Same AOI mask as the automatic workflow
        aoi = gpd.read_file(STUDY_AREA_PATH)

        if aoi.crs != template.crs:
            aoi = aoi.to_crs(template.crs)

        geom = aoi.geometry.union_all()

        inside = geometry_mask(
            [geom.__geo_interface__],
            (template.height, template.width),
            transform=template.transform,
            invert=True,
        )

        resampling_name = template.tags().get(
            "shared_grid_resampling",
            "lanczos",
        )

        resampling = getattr(
            Resampling,
            resampling_name,
        )

        src_nodata = (
            src.nodata
            if src.nodata is not None
            else nodata
        )

        with rasterio.open(
            out_path,
            "w",
            **profile,
        ) as dst:

            for band in range(1, src.count + 1):

                data = np.full(
                    (template.height, template.width),
                    nodata,
                    dtype=profile["dtype"],
                )

                reproject(
                    source=src.read(band),   # <-- IMPORTANT FIX
                    destination=data,
                    src_transform=shifted_transform,
                    src_crs=src.crs,
                    src_nodata=src_nodata,
                    dst_transform=template.transform,
                    dst_crs=template.crs,
                    dst_nodata=nodata,
                    resampling=resampling,
                )

                data[~inside] = nodata
                dst.write(data, band)

            # Preserve metadata from automatic shared product
            dst.update_tags(**template.tags())

            envi_tags = template.tags(ns="ENVI")
            if envi_tags:
                dst.update_tags(ns="ENVI", **envi_tags)

            for band, description in enumerate(
                template.descriptions,
                start=1,
            ):
                if description:
                    dst.set_band_description(
                        band,
                        description,
                    )

            # Manual-coreg provenance
            dst.update_tags(
                coreg_registration_mode="automatic_plus_manual",
                coreg_auto_shift_x_map=dx_auto,
                coreg_auto_shift_y_map=dy_auto,
                coreg_manual_shift_x_map=dx_manual,
                coreg_manual_shift_y_map=dy_manual,
                coreg_final_shift_x_map=dx_final,
                coreg_final_shift_y_map=dy_final,
                coreg_manual_reason=reason,
            )

    results.append({
        "product": product,
        "acquisition": acquisition,
        "x_auto": dx_auto,
        "y_auto": dy_auto,
        "x_manual": dx_manual,
        "y_manual": dy_manual,
        "x_final": dx_final,
        "y_final": dy_final,
        "output": str(out_path),
        "reason": reason,
    })

    print(
        f"✓ {product} | {acquisition}\n"
        f"  automatic: ({dx_auto:+.2f}, {dy_auto:+.2f}) m\n"
        f"  manual:    ({dx_manual:+.2f}, {dy_manual:+.2f}) m\n"
        f"  final:     ({dx_final:+.2f}, {dy_final:+.2f}) m"
    )


manual_summary = pd.DataFrame(results)

if not manual_summary.empty:
    manual_summary.to_csv(
        COREG_DIR_BASE / "manual_coreg_summary.csv",
        index=False,
    )

manual_summary

✓ SIF_SFM | 20240613-PHY-1149-1340-L2-W
  automatic: (-1.24, -0.50) m
  manual:    (-3.00, -4.00) m
  final:     (-4.24, -4.50) m
✓ SIF_iFLD | 20230617-PHY-1124-1360-L2-W
  automatic: (+0.12, -0.50) m
  manual:    (+2.00, +28.00) m
  final:     (+2.12, +27.50) m
✓ SIF_iFLD | 20240613-PHY-1149-1340-L2-W
  automatic: (-7.68, -2.20) m
  manual:    (+3.00, +0.00) m
  final:     (-4.68, -2.20) m


,product,acquisition,x_auto,y_auto,x_manual,y_manual,x_final,y_final,output,reason
0,SIF_SFM,20240613-PHY-1149-1340-L2-W,-1.243378,-0.500000,-3.0,-4.0,-4.243378,-4.500000,E:\Proj1_Pfynwald_Data\HyPlant\coregistration\...,Residual offset after automatic coregistration
1,SIF_iFLD,20230617-PHY-1124-1360-L2-W,0.121504,-0.500000,2.0,28.0,2.121504,27.500000,E:\Proj1_Pfynwald_Data\HyPlant\coregistration\...,Residual offset after automatic coregistration
2,SIF_iFLD,20240613-PHY-1149-1340-L2-W,-7.675612,-2.200257,3.0,0.0,-4.675612,-2.200257,E:\Proj1_Pfynwald_Data\HyPlant\coregistration\...,Residual offset after automatic coregistration
